In [1]:
import scanpy as sc
import anndata as ad
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd 

from pathlib import Path


# Loading

In [2]:
raw_path = "/home/bmb/haxx/working/ceisel_mumm/raws/"
ann_path = "/home/bmb/haxx/working/ceisel_mumm/data/"

In [3]:
! ls {raw_path}

In [4]:
## We need to isolate each set of files into a directory for a scanpy import 

# !mkdir {raw_path}/GSM8287434_RGC_24+48h_ranger
# !mv {raw_path}/GSM8287434_RGC_24+48h_*.gz {raw_path}/GSM8287434_RGC_24+48h_ranger

# !mkdir {raw_path}/GSM8287438_RGC_12+72h_ranger
# !mv {raw_path}/GSM8287438_RGC_12+72h_*.gz {raw_path}/GSM8287438_RGC_12+72h_ranger

In [5]:
annotations = pd.read_csv(ann_path + "ceisel_adata_metadata.csv")
print(annotations.shape)
annotations.head()

In [6]:
# Annotration barcodes combine file name with barcode, we're stripping it out to match the count files

stripped_annotation_ids = [id.split("_")[2] for id in annotations['Cell.ID']]
stripped_annotation_ids
file_assignment = ["_".join(id.split("_")[:2]) for id in annotations['Cell.ID']]
annotations['Stripped.Cell.ID'] = stripped_annotation_ids
annotations['File.Assignment'] = file_assignment
annotations.set_index('Stripped.Cell.ID',inplace=True)

middle_annotations = annotations[annotations['File.Assignment'] == '24_48']
outer_annotations = annotations[annotations['File.Assignment'] == '12_72']
middle_annotations.head()
outer_annotations.head()


In [7]:
middle_raw = sc.read_10x_mtx(
    "/home/bmb/haxx/working/ceisel_mumm/raws/GSM8287434_RGC_24+48h_ranger",
    prefix="GSM8287434_RGC_24+48h_"
)
print(middle_raw.shape)
outer_raw = sc.read_10x_mtx(
    "/home/bmb/haxx/working/ceisel_mumm/raws/GSM8287438_RGC_12+72h_ranger",
    prefix="GSM8287438_RGC_12+72h_"
)
print(outer_raw.shape)

In [8]:
# Not all barcodes that are in the annotation file are also present in the raw file 
# Presumably annotation was created after original filtering

middle_ann_names = set(middle_annotations.index)
matching_middle = [id in middle_ann_names for id in middle_raw.obs_names]
print(f"middle: {np.sum(matching_middle)} out of {middle_raw.shape[0]}")
middle_raw.obs['annotated'] = matching_middle
outer_ann_names = set(outer_annotations.index)
matching_outer = [id in outer_ann_names for id in outer_raw.obs_names]
print(f"outer: {np.sum(matching_outer)} out of {outer_raw.shape[0]}")
outer_raw.obs['annotated'] = matching_outer


In [9]:
# Retain only cells that are present in the annotation file
# Then join the annotations

middle_annotated = middle_raw[middle_raw.obs['annotated']]
outer_annotated = outer_raw[outer_raw.obs['annotated']]
middle_annotated.obs = middle_annotated.obs.join(middle_annotations)
outer_annotated.obs = outer_annotated.obs.join(outer_annotations)
middle_annotated.obs.head()
outer_annotated.obs.head()


In [10]:
joint_raw = ad.concat([middle_annotated,outer_annotated])
joint_raw.shape

# QC On Annotated Data

In [11]:
sc.pp.calculate_qc_metrics(joint_raw, inplace=True)

In [12]:
split_names = [n.split("-") for n in joint_raw.obs_names]
suffix = [n[-1] for n in split_names]

joint_raw.obs["library"] = suffix

plt.figure()
plt.title("Total Counts (All)")
plt.hist(joint_raw.obs["total_counts"],bins=np.arange(0,10000,100))
plt.xlabel("Counts Per Cell")
plt.ylabel("Frequency")
plt.show()

plt.figure()
plt.title("Cells Per Library")
plt.hist(suffix)
plt.xlabel("Library")
plt.ylabel("Frequency")
plt.show()

plt.figure()
plt.title("Total Counts Split by Library")
for suffix in set(suffix):
    masked = joint_raw[joint_raw.obs["library"] == suffix]
    plt.hist(masked.obs["total_counts"],bins=np.arange(800,4000,100),alpha=0.3,label=suffix)
plt.xlabel("Counts Per Cell")
plt.ylabel("Frequency")
plt.legend()
plt.show()

# Retaining Signature Genes

#### Parthanatos

In [13]:
parthanatos_signature_genes = ["Il1b","Aif1","App","Dlg4","Endog","F2","Grin2b","Icam1","Il1b","Map2k1","Mapk1","Mmp9","Parg","Parp1","Plat","S100b","Sirt1"]
parthanatos_signature_genes = [g.lower() for g in parthanatos_signature_genes]
parthanatos_signature_set = set(parthanatos_signature_genes)
parthanatos_signature_set

In [14]:
parthanatos_present = [p for p in parthanatos_signature_set if p in set(joint_raw.var_names)]
parthanatos_missing = [p for p in parthanatos_signature_set if p not in set(joint_raw.var_names)]

print(parthanatos_present)
print(parthanatos_missing)


parthanatos_replacement_addons = [
    'parga','pargl',
    'dlg4a','dlg4b','dlg4b-1',
    'aif1l',
    'appa','appb',
    'grin2bb'
]

# F2 and s100b are present in the ann, but fall below even very conservative filtering thresholds. Removing. 
parthanatos_present.remove('f2')
parthanatos_present.remove('s100b')

print(joint_raw[:,'f2'].var)
print(joint_raw[:,'s100b'].var)

In [15]:
# [g for g in joint_raw.var_names if "f2" in g[:3]]

# icam1: searched for the cd54 alias, searched or all icams and found only icam3
# s100b: questionably present, difficult to analogize? Letter soup present is t,a,w,v,u, and z
# f2 should theoretically be present? Not sure why it's not showing up. Omitting 

parthanatos_translated_signature = parthanatos_present + parthanatos_replacement_addons
parthanatos_translated_signature


#### Apoptosis

In [16]:
apoptosis_signature_genes = ["Bax","Cycs","Akt1","Akt1","Bax","Bcl2l1","Birc2","Cycs","Ikbkb","Irak1","Pik3ca","Pik3r1","Pik3r2","Ppp3ca","Ppp3cb","Ppp3r1","Prkaca","Prkacb","Prkar1a","Prkar1b","Xiap"]
apoptosis_signature_genes = [g.lower() for g in apoptosis_signature_genes]
apoptosis_signature_set = set(apoptosis_signature_genes)

apoptosis_present = [p for p in apoptosis_signature_set if p in set(joint_raw.var_names)]
apoptosis_missing = [p for p in apoptosis_signature_set if p not in set(joint_raw.var_names)]

print(apoptosis_present)
print(apoptosis_missing)

In [17]:
# Same as above, cycsa is gettinf filtered out, so I'm omitting. 
apoptosis_replacement_addons = [
    'baxb','baxa','baxa-1',
    'cycsb', # 'cycsa',
    'prkacaa','prkacab',
    'prkacba','prkacbb',
    'prkar1aa','prkar1ab','prkar1b',
    'ppp3r1b', 'ppp3r1a'
]

In [18]:
# Manual replacement search 
print([g for g in joint_raw.var_names if "birc2" in g])

# baxa,baxb,baxa-1 for bax

apoptosis_translated_signature = apoptosis_present + apoptosis_replacement_addons
apoptosis_translated_signature


#### Necroptosis General(?)

In [19]:
# These duplicates are an exact copy from the xls from the paper. Why? I don't know. The coefficients they claim appear duplicated so I don't think it's affecting their analysis?
necroptosis_signature_genes = ["Pgam5","Ripk3","Hmgb1","Dnm1l","Hmgb1","Mlkl","Pgam5","Ripk1","Ripk3","Mlkl","Ripk1"] 
necroptosis_signature_genes = [g.lower() for g in necroptosis_signature_genes]
necroptosis_signature_set = set(necroptosis_signature_genes)

necroptosis_present = [p for p in necroptosis_signature_set if p in set(joint_raw.var_names)]
necroptosis_missing = [p for p in necroptosis_signature_set if p not in set(joint_raw.var_names)]

print(necroptosis_present)
print(necroptosis_missing)


In [20]:
print([g for g in joint_raw.var_names if "flj" in g])

# Mlkl appears wholly missing. The only alias I see online is flj34389, which is also absent (very few fljs generally)
necroptosis_replacement_addons = [
    'hmgb1a','hmgb1b',
    'ripk1l',
]

# HMGB1A & B are gregarious in development, we can mask them and see what happens

necroptosis_translated_signature = necroptosis_present + necroptosis_replacement_addons
necroptosis_translated_signature

In [21]:
# necroptosis_translated_signature = ['pgam5', 'ripk3', 'dnm1l', 'ripk1l'] # Kicking out the dev

# Hand-curated instead, attr Ceisel
necroptosis_translated_signature = [
    'ripk1l','ripk3','fadd',
    'cyldb','cylda','cyldl',
    'igfbp2a','igfbp2b',
    'pgam5','tnfrsf1a','vdac1','vdac2','rbck1','tradd','dnm1l']

In [22]:
population_marker_sets = {
    'cones': ['gnat2','arr3'],
    'rods': ['rho','gnat1','gngt1','sagb','pde6a','pde6b','nrl'],
    'muller_glia':['rlbp1a','rx1','crabp1a','gpr37a',],
    'rgcs':['rbpms2b','isl2b','uchl1','rbpms2a','pou4f2'],
    'bipolar_cells':['nrn1lb','cabp5a','vsx1','calb2b','vamp1','rs1a','syt5b',],
    'amacrine_cells':['pax6a','pax6b',],
    'horizontal_cells':['slc4a5b','tnr','rem1','CR361564.1','cx52.9','onecut','cx55.5'],
}
all_cell_markers = [g for l in population_marker_sets.values() for g in l]
all_cell_markers

In [23]:
candidates = {}
for marker in all_cell_markers:
    candidates[marker] = [g for g in joint_raw.var_names if marker in g]
candidates

In [24]:
print([g for g in middle_raw.var_names if "tag" in g])

In [25]:
bipolar_markers = ['nrn1lb','cabp5a','vsx1','calb2b','vamp1','rs1a','syt5b',]

# Filtering For Salience

In [26]:
sc.pp.filter_genes(joint_raw, min_cells=1)
sc.pp.filter_cells(joint_raw, min_counts=1)

copy = joint_raw.copy()
sc.pp.log1p(copy)
sc.pp.highly_variable_genes(copy, n_top_genes=5000)

# We will filter on highly variable so let's just mark our targets as hv
copy.var['highly_variable'].loc[parthanatos_translated_signature] = True
copy.var['highly_variable'].loc[necroptosis_translated_signature] = True
copy.var['highly_variable'].loc[apoptosis_translated_signature] = True
copy.var['highly_variable'].loc[bipolar_markers] = True

joint = joint_raw[:,copy.var["highly_variable"]].copy()

joint.var['parthanatos'] = [g in parthanatos_translated_signature for g in joint.var_names]
joint.var['necroptosis'] = [g in necroptosis_translated_signature for g in joint.var_names]
joint.var['apoptosis'] = [g in apoptosis_translated_signature for g in joint.var_names]


# sc.pp.downsample_counts(joint,counts_per_cell=1000)
sc.pp.normalize_total(joint,exclude_highly_expressed=True)
joint.layers['raw'] = joint.X.copy()
sc.pp.log1p(joint)

In [27]:
print("Computing PCA")
sc.pp.pca(joint, n_comps=50)
print("Computing neighbors")
sc.pp.neighbors(joint)
print("Computing UMAP")
sc.tl.umap(joint)
sc.pl.umap(joint)


In [28]:
log_total_counts = np.log1p(joint.obs['total_counts'])
joint.obs['log_total_counts'] = log_total_counts
sc.pl.umap(joint, color=["log_total_counts"])
sc.pl.umap(joint, color=["library"])

In [29]:
time = [c.split("_")[0].strip("h") for c in joint.obs['Condition']]
condition = ["_".join(c.split("_")[1:]) for c in joint.obs['Condition']]
# set(condition)
joint.obs['exp_condition'] = condition
joint.obs['time'] = np.array(time).astype(int)
# joint.obs
# sc.pl.umap(joint, color=["time"])

# joint.obs['Condition']

In [30]:
data_path = "/home/bmb/haxx/working/ceisel_mumm/data/"
sc.write(data_path + "joint_annotated.h5ad",joint)

In [31]:

plt.figure()
plt.title("Log total counts vs time")
plt.boxplot(
    [joint.obs['log_total_counts'][joint.obs['time'] == t] for t in [12,24,48,72]],
    showfliers=False,
)
plt.xticks(
    [1,2,3,4],
    ["12h","24h","48h","72h"]
)
plt.ylabel("Log total counts")
plt.show()



# Cell Type Annotation

In [32]:
data_annotated_path = "/home/bmb/haxx/working/ceisel_mumm/data/"
data = sc.read_h5ad(data_annotated_path + "joint_annotated.h5ad")


In [33]:
sc.tl.leiden(data, resolution=1)
sc.pl.umap(data, color=['leiden'],legend_loc='on data')

In [34]:
# sc.tl.rank_genes_groups(data, 'leiden')
# sc.pl.rank_genes_groups(data, n_genes=25)

In [35]:
population_marker_sets = {
    'cones': ['gnat2','arr3a','arr3b'],
    'rods': ['rho','gnat1','gngt1','sagb','pde6a','pde6b','nrl'],
    'muller_glia':['rlbp1a','rx1','crabp1a','gpr37a',],
    'rgcs':['rbpms2b','isl2b','uchl1','rbpms2a','pou4f2'],
    'rgc_secondary':['nrn1a','isl2b','islr2',],
    'microglia':['mfap4','mpeg1.1','csf1ra','spi1b'],
    'bipolar_cells':['nrn1lb','cabp5a','vsx1','calb2b','vamp1','rs1a','syt5b',], # It looks like the ENTIRETY of bipolar cells got filtered?
    'amacrine_cells':['pax6b','slc6a1b','pax10','nhlh2','pvalb6'], # pax6a got filtered
    'horizontal_cells':['onecutl','tnr','rem1','CR361564.1','cx52.9','ompa','opn9'], # onecut ambiguous, cx55.5, slc4a5b seem fully missing from the index
    'undifferentiated':['sox2','olig2','hes6','hmgn2','hmga1a','atoh7'],
    'pr_precursors':['tulp1a','opn6a','tmem244']
}

In [36]:
sc.pl.dotplot(data,population_marker_sets,groupby="leiden",figsize=(10,8))

In [41]:
mapping = {
    k:k for k in data.obs['leiden'].unique()
}

mapping |= {
    '0': 'Bipolar cells', 
    '1': 'Bipolar cells',
    '2': 'Bipolar cells',
    '3': 'Bipolar cells',
    '4': 'Amacrine cells',
    # 5
    '6': 'Cones',
    # 7
    '8': 'Horizontal cells',
    '9': 'Muller glia',
    '10': 'Horizontal cells',
    # 11
    '12': 'Cones',
    '13': 'Bipolar cells',
    '14': 'Bipolar cells',
    '15': 'Bipolar cells',
    '16': 'Amacrine cells',
    '17': 'RGCs', 
    '18': 'Bipolar cells',
    '19': 'Undifferentiated',
    '20': 'Cones', # Probably immature
    '21': 'Bipolar cells',
    '22': 'Bipolar cells',#small
    # 23
    '24': 'PR precursors', 
    '25': 'Rods',
    # 26
    # 27
    '28': 'Microglia',
    # 29
    '30': 'Horizontal cells', # Probably immature
    '31': 'trash',

}
# mapping
# Create a new column with renamed clusters
data.obs['cell_type'] = data.obs['leiden'].map(mapping).astype('category')


In [42]:
sc.pl.dotplot(data,population_marker_sets,groupby="cell_type",figsize=(10,8))

In [43]:
sc.pl.umap(data,color='cell_type',legend_loc='on data')

In [47]:
sc.write(data_annotated_path + "full_annotations_leiden.h5ad",data)

In [49]:
parthanatos_mask = np.array(data.var['parthanatos'])
apoptosis_mask = np.array(data.var['apoptosis'])
necroptosis_mask = np.array(data.var['necroptosis'])

parthanatos_signature = data.var_names[parthanatos_mask]
apoptosis_signature = data.var_names[apoptosis_mask]
necroptosis_signature = data.var_names[necroptosis_mask]

print(parthanatos_signature)
print(necroptosis_signature)
print(apoptosis_signature)

sc.tl.score_genes(data,parthanatos_signature,score_name="parthanatos")
sc.tl.score_genes(data,necroptosis_signature,score_name="necroptosis")
sc.tl.score_genes(data,apoptosis_signature,score_name="apoptosis")

In [54]:
data.obs['time'] = data.obs['time'].astype(str)


In [ ]:
# sc.pl.dotplot(data,parthanatos_signature,groupby="cell_type")
rgc_subset = data[data.obs['cell_type'] == 'RGCs',parthanatos_signature]
injury_mask = np.array(rgc_subset.obs['exp_condition'] == 'Mtz')
time_sort_injury = np.argsort(rgc_subset[injury_mask].obs['time'].astype(int))
time_sort_control = np.argsort(rgc_subset[~injury_mask].obs['time'].astype(int))

rgc_matrix = rgc_subset.X.toarray()
injury_matrix = rgc_matrix[injury_mask][time_sort_injury]
control_matrix = rgc_matrix[~injury_mask][time_sort_control]

fig,(ax0,ax1) = plt.subplots(1,2)
im0 = ax0.imshow(
    injury_matrix,
    cmap='viridis',
    aspect='auto',
    interpolation='nearest',vmin=0,vmax=1.5
)
im1 = ax1.imshow(
    control_matrix,
    cmap='viridis',
    aspect='auto',
    interpolation='nearest',vmin=0,vmax=1.5
)
ax0.set_title("Injury")
ax1.set_title("Control")
plt.colorbar(im0,ax=ax0)
plt.colorbar(im1,ax=ax1)
fig.tight_layout()
fig.show()

In [ ]:
# Annotation 
# - Timecourse 
# - Cell type (especially target (RGC/rods)) 
#      * microglia doublet warning here
# - Can we basically establish partial death, then regeneration 
# - peak death at 48h
# - regen starting at 72 (proliferation is starting)
# - We don't expect a developmental trajectory in general
# - PCA/ICA <- 

# - PCA delta between untreated treated
# - project onto apoptosis and necroptosis (and parthanatos if possible)

# - GSEA on raw deltas 